# Week 8: Vector DB / Retrieval

SiteLens AI — Inspection-precedent retrieval prototype.

**Topic:** NLP, vector databases, RAG  
**Dates:** May 10–14 2026  
**Deliverable:** Pinecone + sentence-transformers retrieval prototype for inspection precedents.

**Status:** In progress.

In [ ]:
# Week 8 implementation

In [ ]:
%pip install pinecone sentence-transformers python-dotenv




In [ ]:
%pip install pinecone sentence-transformers python-dotenv

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

load_dotenv("../.env")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "sitelens")

assert PINECONE_API_KEY, "PINECONE_API_KEY not loaded — check .env path"
print("Key loaded:", PINECONE_API_KEY[:8], "...")

In [ ]:
GPKG_PATH = "../data/raw/Noto_Peninsula_Damage_2_5.gpkg"

MMI_LABELS = [(9, "violent"), (8, "severe"), (7, "very strong"),
              (6, "strong"), (4, "moderate"), (0, "light")]

def mmi_label(mmi):
    for threshold, label in MMI_LABELS:
        if mmi >= threshold:
            return label
    return "light"

DAMAGE_TEXT = {0: "survived", 1: "destroyed", 9: "obstructed", 99: "inconsistent footprint"}

def row_to_text(row):
    outcome = DAMAGE_TEXT.get(row.damage_val, "unknown")
    hazards = [h for h, flag in [("fire", row.GSI_fire), ("tsunami", row.GSI_tsunami),
               ("slope failure", row.GSI_slope_failure)] if flag == 1]
    haz_str = ", ".join(hazards) if hazards else "seismic only"
    mmi_str = f"MMI {row.USGS_MMI:.1f} ({mmi_label(row.USGS_MMI)} shaking)"
    loc = row.municipality if pd.notna(row.municipality) else "unknown municipality"
    return (f"Building {outcome}. Hazard: {haz_str}. {mmi_str}. "
            f"Location: {loc}. Evidence: {row.conf}-source assessment.")

gdf_d = gpd.read_file(GPKG_PATH, layer="v2.5", where="damage_val = 1").head(12)
gdf_s = gpd.read_file(GPKG_PATH, layer="v2.5", where="damage_val = 0").head(6)
gdf_o = gpd.read_file(GPKG_PATH, layer="v2.5", where="damage_val = 9").head(2)
gdf   = pd.concat([gdf_d, gdf_s, gdf_o], ignore_index=True)

records = [{"id": f"bldg_{i:04d}", "text": row_to_text(row)}
           for i, (_, row) in enumerate(gdf.iterrows())]

for r in records[:3]:
    print(r["text"])


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([r["text"] for r in records], show_progress_bar=True)
print(f"Shape: {embeddings.shape}")


In [ ]:
pc.delete_index(INDEX_NAME)
pc.create_index(
    name=INDEX_NAME,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
print(index.describe_index_stats())
index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())


In [ ]:
vectors = [
    {"id": records[i]["id"],
     "values": embeddings[i].tolist(),
     "metadata": {"text": records[i]["text"], "damage_val": int(gdf.iloc[i].damage_val)}}
    for i in range(len(records))
]
index.upsert(vectors=vectors)
print(f"Upserted {len(vectors)} vectors")


In [ ]:
import time
time.sleep(5)

queries = [
    "building destroyed by fire in dense urban area",
    "structure survived tsunami zone with strong shaking",
]

for query in queries:
    vec = model.encode([query])[0].tolist()
    results = index.query(vector=vec, top_k=3, include_metadata=True)
    print(f"\nQuery: '{query}'")
    for m in results["matches"]:
        print(f"  [{m['score']:.3f}] {m['id']}  {m['metadata']['text']}")



In [ ]:
import os
print(os.getcwd())